
# Lab 1 - Introduction to LLMs: Practical Activities

**Student:** Jatin Kolhapure  
**Roll No:** SCFP125014  
**Subject:** Generative AI

## Activity 1 — Explore LLM Models on Hugging Face

### Objective
Explore three Hugging Face models: one small, one medium, and one large.
Record their creators, parameter counts, generative capability, context length,
local execution suitability, and recommended hardware.

### Step 1.1 — Import the required library


In [ ]:

# Activity 1 - Step 1.1
# Import the Hugging Face API client to retrieve model metadata.

from huggingface_hub import HfApi

api = HfApi()
print("Hugging Face API client initialized.")


Hugging Face API client initialized.



### Step 1.2 - Retrieve model metadata

The following code requests information for the three selected models.
It retrieves metadata only; it does not download or execute the models.


In [ ]:

# Activity 1 - Step 1.2
# Retrieve model metadata from Hugging Face.
# Missing fields are reported as unavailable rather than guessed.

models_to_explore = [
    ("Small", "distilbert/distilgpt2"),
    ("Medium", "openai-community/gpt2-medium"),
    ("Large", "meta-llama/Llama-3.2-1B"),
]

for size, model_id in models_to_explore:
    print("\n" + "=" * 70)
    print(f"Category: {size}")
    print(f"Requested model: {model_id}")

    try:
        info = api.model_info(model_id)

        print("Model ID:", getattr(info, "id", "Not available"))
        print("Author:", getattr(info, "author", "Not available"))
        print("Pipeline tag:", getattr(info, "pipeline_tag", "Not available"))
        print("Library:", getattr(info, "library_name", "Not available"))
        print("Downloads:", getattr(info, "downloads", "Not available"))

        # Display only selected configuration fields when available.
        config = getattr(info, "config", None) or {}
        if isinstance(config, dict):
            print("Model architecture:", config.get("architectures", "Not available"))
            print("Model type:", config.get("model_type", "Not available"))
            print("Maximum positions:", config.get("max_position_embeddings", "Not available"))
        else:
            print("Model configuration:", config)

        # Keep the model card data available for further inspection.
        card_data = getattr(info, "cardData", None)
        print("Model card data available:", card_data is not None)

    except Exception as error:
        print("Metadata retrieval failed.")
        print("Error type:", type(error).__name__)
        print("Error details:", str(error))

print("\nMetadata inspection finished.")



Category: Small
Requested model: distilbert/distilgpt2
Model ID: distilbert/distilgpt2
Author: distilbert
Pipeline tag: text-generation
Library: transformers
Downloads: 1796075
Model architecture: ['GPT2LMHeadModel']
Model type: gpt2
Maximum positions: Not available
Model card data available: True

Category: Medium
Requested model: openai-community/gpt2-medium
Model ID: openai-community/gpt2-medium
Author: openai-community
Pipeline tag: text-generation
Library: transformers
Downloads: 174673
Model architecture: ['GPT2LMHeadModel']
Model type: gpt2
Maximum positions: Not available
Model card data available: True

Category: Large
Requested model: meta-llama/Llama-3.2-1B
Model ID: meta-llama/Llama-3.2-1B
Author: meta-llama
Pipeline tag: text-generation
Library: transformers
Downloads: 573597
Model architecture: ['LlamaForCausalLM']
Model type: llama
Maximum positions: Not available
Model card data available: True

Metadata inspection finished.



### Step 1.3 - Inspect model configuration and parameter counts

The previous output confirmed the model IDs, authors, task tags, and architectures.
This step checks the model configuration and parameter count separately so the
results are easier to inspect and record.


In [ ]:

# Activity 1 - Step 1.3
# Retrieve configuration details and count parameters from model metadata.
# This does not download the model weights.

for size, model_id in models_to_explore:
    print("\n" + "=" * 60)
    print(f"Category: {size}")
    print(f"Model: {model_id}")

    try:
        info = api.model_info(model_id, files_metadata=False)
        config = getattr(info, "config", None) or {}
        card = getattr(info, "cardData", None) or {}

        print("Author:", getattr(info, "author", "Not available"))
        print("Task:", getattr(info, "pipeline_tag", "Not available"))
        print("Architecture:", config.get("architectures", "Not available"))
        print("Model type:", config.get("model_type", "Not available"))

        # Some model cards publish parameter counts in their metadata.
        params = card.get("parameter_count") if isinstance(card, dict) else None
        print("Parameter count in card metadata:", params or "Not listed in this field")

        # Show configuration keys that may help verify context length.
        if isinstance(config, dict):
            relevant_keys = [
                key for key in config
                if any(term in key.lower() for term in ["position", "context", "sequence", "token"])
            ]
            if relevant_keys:
                for key in relevant_keys:
                    print(f"{key}:", config[key])
            else:
                print("Context-related configuration keys: Not found in this response")

    except Exception as error:
        print("Inspection failed:", type(error).__name__, str(error))



Category: Small
Model: distilbert/distilgpt2
Author: distilbert
Task: text-generation
Architecture: ['GPT2LMHeadModel']
Model type: gpt2
Parameter count in card metadata: Not listed in this field
tokenizer_config: {}

Category: Medium
Model: openai-community/gpt2-medium
Author: openai-community
Task: text-generation
Architecture: ['GPT2LMHeadModel']
Model type: gpt2
Parameter count in card metadata: Not listed in this field
tokenizer_config: {}

Category: Large
Model: meta-llama/Llama-3.2-1B
Author: meta-llama
Task: text-generation
Architecture: ['LlamaForCausalLM']
Model type: llama
Parameter count in card metadata: Not listed in this field
tokenizer_config: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>'}


In [4]:
# Step 1.4: Inspect parameter-count and context-length fields
# This checks what the Hugging Face API actually returns.

for size, model_id in models_to_explore:
    print("\n" + "=" * 60)
    print(f"Category: {size}")
    print(f"Model: {model_id}")

    try:
        info = api.model_info(model_id)
        config = getattr(info, "config", None) or {}

        if not isinstance(config, dict):
            config = {}

        print("Model ID:", getattr(info, "id", "Not available"))
        print("Author:", getattr(info, "author", "Not available"))

        # Look for configuration keys related to context length.
        context_keys = [
            key for key in config
            if any(word in key.lower()
                   for word in ["position", "context", "sequence", "token"])
        ]

        if context_keys:
            for key in context_keys:
                print(f"{key}: {config[key]}")
        else:
            print("Context length: Not available in inspected configuration")

        # The API may not expose parameter count directly.
        print("Parameter count: Check the model card")

    except Exception as error:
        print("Error:", type(error).__name__, str(error))


Category: Small
Model: distilbert/distilgpt2
Model ID: distilbert/distilgpt2
Author: distilbert
tokenizer_config: {}
Parameter count: Check the model card

Category: Medium
Model: openai-community/gpt2-medium
Model ID: openai-community/gpt2-medium
Author: openai-community
tokenizer_config: {}
Parameter count: Check the model card

Category: Large
Model: meta-llama/Llama-3.2-1B
Model ID: meta-llama/Llama-3.2-1B
Author: meta-llama
tokenizer_config: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>'}
Parameter count: Check the model card


### Step 1.5 - Comparison of Small, Medium, and Large Language Models

The three models were inspected using the Hugging Face API. The API returned each model's ID, author, task, and architecture. Parameter counts and context lengths were checked against the model documentation because they were not available in the inspected API fields.

| Feature | Small Model | Medium Model | Large Model |
|---|---|---|---|
| Model name | DistilGPT2 | GPT-2 Medium | Llama 3.2 1B |
| Hugging Face model ID | `distilbert/distilgpt2` | `openai-community/gpt2-medium` | `meta-llama/Llama-3.2-1B` |
| Developer | Hugging Face | OpenAI | Meta |
| Parameters | 82 million | 355 million | Approximately 1.23 billion |
| Generative model? | Yes | Yes | Yes |
| Context length | 1,024 tokens | 1,024 tokens | Up to 128,000 tokens for the standard text model |
| Local execution | Easily suitable for a typical laptop | Usually suitable on a laptop with sufficient RAM | Possible on a capable laptop; quantization can reduce memory requirements |
| Recommended hardware | CPU and 4 GB or more available RAM | CPU and 8 GB or more available RAM; GPU can improve speed | Prefer a GPU with sufficient VRAM or use a quantized model; memory needs depend on precision and context length |

**Observations**

1. DistilGPT2 is the smallest model and is suitable for basic local text-generation experiments.
2. GPT-2 Medium has more parameters than DistilGPT2 and generally requires more memory for inference.
3. Llama 3.2 1B is substantially larger and supports a much longer context window in its standard text-model configuration.
4. The actual memory requirement depends on model precision, context length, and inference software. The hardware recommendations above are practical estimates, not results measured by the API.

**Conclusion**

The inspection demonstrated that Hugging Face provides model metadata, while some details—such as parameter count and context length—may require consulting the model documentation. Model size and available hardware should be considered when choosing a model for local execution.

### Activity 1 - Answers to the Questions

**1. Who created the model?**

- **Small — DistilGPT2:** The model repository is maintained under `distilbert` on Hugging Face.
- **Medium — GPT-2 Medium:** The original model was developed by OpenAI; the repository used in this experiment is `openai-community/gpt2-medium`.
- **Large — Llama 3.2 1B:** Developed by Meta.

**2. How many parameters does it have?**

- DistilGPT2: approximately 82 million parameters.
- GPT-2 Medium: approximately 355 million parameters.
- Llama 3.2 1B: approximately 1.23 billion parameters.

**3. Is it a generative model?**

Yes. All three are autoregressive text-generation models that predict subsequent tokens to generate text.

**4. What is the context length?**

- DistilGPT2: 1,024 tokens.
- GPT-2 Medium: 1,024 tokens.
- Llama 3.2 1B: up to 128,000 tokens for the standard text model.

The context length represents the amount of tokenized input and context the model can process, subject to its configuration and implementation.

**5. Is the model suitable for local execution?**

- **DistilGPT2:** Yes. Its small size makes it suitable for experiments on a typical laptop.
- **GPT-2 Medium:** Yes, on a laptop with sufficient RAM; CPU inference may be slower.
- **Llama 3.2 1B:** Yes, on compatible hardware. Quantization can reduce memory requirements.

**6. What hardware does the model card recommend?**

The metadata inspection performed in this experiment did not return specific hardware recommendations. As a practical guideline, DistilGPT2 can run on a CPU, GPT-2 Medium benefits from sufficient RAM and potentially a GPU, and Llama 3.2 1B can benefit from a GPU or a quantized configuration when memory is limited.

### Result

Three Hugging Face text-generation models were inspected and compared by creator, parameter count, generative capability, context length, local execution suitability, and hardware considerations.

## Activity 2 — Understanding Tokenization

### Objective
To understand how a tokenizer converts human-readable text into tokens and numerical token IDs that a language model can process.

### Explanation
Tokenization is the process of splitting text into smaller units called tokens. Depending on the tokenizer, a token can represent a word, part of a word, punctuation, or another text unit. Each token is mapped to a numerical ID. A tokenizer can also decode token IDs back into text. <Cite refs={["turn200438search4","turn200438search0"]} />

### Step 2.2 - Load the pretrained tokenizer

In [5]:
# Step 2.2: Load a pretrained tokenizer

from transformers import AutoTokenizer

model_name = "distilbert/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully.")
print("Model:", model_name)

d:\Mit\Sem 3\GenAI\SCFP125014_MCA_3rd_Sem\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer loaded successfully.
Model: distilbert/distilgpt2


### Step 2.3 - Convert text into tokens and token IDs

In [6]:
# Step 2.3: Tokenize a sample sentence

text = "Large language models can generate human-like text."

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text, add_special_tokens=False)

print("Original text:")
print(text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

print("\nNumber of tokens:", len(token_ids))

Original text:
Large language models can generate human-like text.

Tokens:
['Large', 'Ġlanguage', 'Ġmodels', 'Ġcan', 'Ġgenerate', 'Ġhuman', '-', 'like', 'Ġtext', '.']

Token IDs:
[21968, 3303, 4981, 460, 7716, 1692, 12, 2339, 2420, 13]

Number of tokens: 10


### Step 2.4 - Convert token IDs back into text

In [7]:
# Step 2.4: Decode token IDs back into text

decoded_text = tokenizer.decode(token_ids)

print("Original text:")
print(text)

print("\nDecoded text:")
print(decoded_text)

Original text:
Large language models can generate human-like text.

Decoded text:
Large language models can generate human-like text.


### Step 2.5 - Observation and Result

**Observation:**

The pretrained DistilGPT2 tokenizer successfully converted the input sentence into tokens and numerical token IDs. The sentence was divided into 10 tokens. The hyphen in `human-like` was tokenized separately, and the `Ġ` marker appeared before tokens that begin with a space.

The token IDs were decoded back into text, and the decoded output matched the original sentence.

**Result:**

Tokenization and decoding were performed successfully using the DistilGPT2 tokenizer. This experiment demonstrates how text is converted into numerical representations that can be processed by a language model.

### Activity 2 - Answers to the Questions

**1. Is one word always one token?**

No. One word does not always correspond to one token. Depending on the tokenizer and vocabulary, a word may be represented by one token or split into multiple tokens. For example, in this experiment, `human-like` was represented by separate tokens for `human`, `-`, and `like`.

**2. Why are some words split?**

A tokenizer uses a predefined vocabulary and tokenization rules. Words or word parts that are not represented as a single token may be divided into smaller units. This allows the model to process a wide variety of words without requiring a separate vocabulary entry for every possible word.

**3. What happens to punctuation?**

Punctuation can be represented as separate tokens. In this experiment, the hyphen (`-`) and the final period (`.`) appeared as separate tokens.

**4. Why does the model need token IDs?**

A language model processes numerical representations rather than raw text characters directly. The tokenizer maps each token to an integer ID, which the model uses to look up token embeddings and perform computations. Decoding maps the resulting token IDs back into readable text.

### Experimental Result

The sentence `Large language models can generate human-like text.` was converted into 10 tokens and their corresponding numerical IDs using the DistilGPT2 tokenizer. Decoding the token IDs reproduced the original sentence successfully.

## Activity 3 - Running an LLM Using Google Colab GPU

### Objective
To run a pretrained language model in Google Colab using GPU acceleration, generate text, and observe GPU availability, memory usage, and generated output length.

### Explanation
Google Colab provides a hosted Python environment and may provide access to a GPU. GPU acceleration can speed up language-model inference compared with CPU execution. Actual performance depends on the selected GPU, model size, precision, and workload.

### Step 3.3 - GPU Verification Result

- **Environment:** Google Colab
- **PyTorch version:** 2.11.0+cu130
- **CUDA available:** True
- **GPU name:** NVIDIA Tesla T4
- **Total GPU memory reported:** 14.56 GB

**Observation:** PyTorch successfully detected CUDA availability and identified the Tesla T4 GPU. Therefore, the current Google Colab runtime is ready for GPU-accelerated language-model inference.

**Result:** GPU availability was verified successfully.

### Step 3.4 - Execute the Language Model on Google Colab

The pretrained DistilGPT2 model (`distilbert/distilgpt2`) was loaded in Google Colab and executed using the NVIDIA Tesla T4 GPU.

**Input prompt:**

Artificial intelligence is changing the world because

**Generated output:**

Artificial intelligence is changing the world because of the way we think about it.

**Execution environment:**
- Platform: Google Colab
- GPU: NVIDIA Tesla T4
- CUDA available: True
- Reported GPU memory: 14.56 GB

### Step 3.5 - Observation and Result

**Observation:**

The DistilGPT2 model successfully generated text from the given prompt in Google Colab using GPU acceleration. The GPU was detected successfully by PyTorch, confirming that the runtime was configured for GPU-based inference.

**Result:**

The pretrained language model was executed successfully in Google Colab, and text generation was demonstrated using the Tesla T4 GPU.

### Step 3.6 - GPU Memory and Output Length Measurements

**Experimental environment:** Google Colab

**GPU:** NVIDIA Tesla T4

| Measurement | Actual Result |
|---|---:|
| Input token count | 8 |
| Total output token count | 58 |
| New tokens generated | 50 |
| Peak GPU memory allocated | 325.57 MB |

**Observation:**

The input prompt was converted into 8 tokens. The model generated 50 new tokens, resulting in 58 tokens in total, including the input. The peak GPU memory allocated by PyTorch during generation was 325.57 MB.

**Note:** The peak memory value represents PyTorch's allocated GPU memory during the measured generation step, not the total memory consumed by the complete Colab runtime.

### Activity 3 - Answers to the Questions

**1. Is the GPU actually being used?**

Yes. PyTorch reported `CUDA available: True`, and the detected GPU was an NVIDIA Tesla T4. The model and input tensors were moved to CUDA using `.to("cuda")`. The successful text-generation experiment and GPU memory measurement confirm that GPU-based inference was performed.

**2. How much memory does the model consume?**

The measured peak GPU memory allocated by PyTorch during the generation experiment was **325.57 MB**. This is the observed allocation reported by PyTorch during the measured operation, not the total GPU memory usage of the entire Colab runtime. The GPU had 14.56 GB of total memory available.

**3. What happens if you increase the output length?**

Increasing the maximum number of generated tokens generally increases generation time because the model must perform additional generation steps. It can also increase memory usage because more tokens and their associated intermediate data must be handled. The exact change depends on the model, context length, and generation settings.

In this experiment, the model generated 50 new tokens. A comparison using different values, such as 20, 50, and 100 new tokens, would be needed to measure the actual differences in generation time and memory consumption.

**4. Would this run on a laptop without a GPU?**

Yes. A small model such as DistilGPT2 can run on a laptop CPU if sufficient system RAM is available. However, generation may be slower than on a suitable GPU. The actual speed depends on the laptop's processor, available memory, and model configuration. Larger models generally require more resources and may be impractical on a low-memory laptop.

### Conclusion

The DistilGPT2 model was successfully executed using an NVIDIA Tesla T4 GPU in Google Colab. GPU availability was verified, text was generated, and peak GPU memory allocation was measured at 325.57 MB. The experiment demonstrates how a cloud GPU can support language-model inference without requiring a dedicated GPU in the local laptop.

### Activity 3 - Output Length Experiment

**Objective:** To observe how increasing the requested output length affects generation time and GPU memory allocation.

**Experimental environment:** Google Colab with NVIDIA Tesla T4 GPU.

| Requested new tokens | Actual new tokens | Total output tokens | Generation time (seconds) | Peak GPU memory (MB) |
|---:|---:|---:|---:|---:|
| 20 | 20 | 28 | 0.158 | 324.52 |
| 50 | 50 | 58 | 0.395 | 325.57 |
| 100 | 100 | 108 | 0.804 | 327.33 |

**Observation:**

1. The model generated the requested number of new tokens in all three tests.
2. Generation time increased from 0.158 seconds for 20 new tokens to 0.804 seconds for 100 new tokens.
3. Peak allocated GPU memory increased slightly, from 324.52 MB to 327.33 MB.
4. The generated text was repetitive and did not meaningfully expand despite the increased token limit. This shows that a higher output limit does not necessarily produce more informative content.

**Answer to Question 3 - What happens if you increase the output length?**

Increasing the output length increased generation time in this experiment. Peak GPU memory allocation also increased slightly. The model generated more tokens, but the additional output consisted mainly of whitespace after the same sentence.

**Note:** Generation time and memory measurements can vary between runs because of runtime conditions and GPU execution behavior.

## Activity 4 — Can My Computer Run an LLM?

### Objective
To inspect the computer's available hardware, estimate memory requirements for language models of different sizes and numerical precisions, and determine which models are practical to run on a computer with limited memory.

### Explanation
Running a language model requires memory for its model weights and additional resources for inference. The required memory depends on the number of parameters, numerical precision, context length, and inference framework. Quantization reduces the number of bits used to represent model weights and can lower memory requirements.

In [8]:
# Step 4.2: Inspect local computer hardware

import platform
import psutil
import torch

print("Operating system:", platform.platform())
print("Processor:", platform.processor())

memory = psutil.virtual_memory()

print("\n--- System RAM ---")
print("Total RAM:", round(memory.total / (1024 ** 3), 2), "GB")
print("Available RAM:", round(memory.available / (1024 ** 3), 2), "GB")

print("\n--- GPU ---")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory
    print("Total GPU memory:", round(total_vram / (1024 ** 3), 2), "GB")
else:
    print("CUDA GPU not detected by PyTorch.")

Operating system: Windows-11-10.0.26200-SP0
Processor: Intel64 Family 6 Model 154 Stepping 3, GenuineIntel

--- System RAM ---
Total RAM: 15.68 GB
Available RAM: 1.06 GB

--- GPU ---
CUDA available: False
CUDA GPU not detected by PyTorch.


### Step 4.3 — Hardware Inspection Results

### Step 4.3 — Hardware Inspection Results

- **Operating system:** Windows-11-10.0.26200-SP0
- **Processor:** Intel64 Family 6 Model 154 Stepping 3, GenuineIntel
- **Total system RAM:** 15.68 GB
- **Available system RAM:** 1.91 GB
- **CUDA GPU available:** False
- **GPU name and memory:** CUDA GPU not detected by PyTorch.

**Observation:** The computer's hardware specifications were inspected using Python. These results will be used to estimate which language models can be run practically on the local machine.

### Step 4.4 - Memory Requirements at Different Model Sizes and Precisions

A model's approximate weight memory can be estimated using the number of parameters and the number of bits used to represent each parameter.

- FP16 uses approximately 2 bytes per parameter.
- INT4 uses approximately 0.5 bytes per parameter.

These estimates cover model weights only. Actual inference also requires memory for the runtime, activations, the key-value (KV) cache, and other overhead.

In [9]:
# Step 4.4: Estimate model weight memory

models = [
    ("4B FP16", 4, 2),
    ("4B INT4", 4, 0.5),
    ("7B INT4", 7, 0.5),
]

print(f"{'Model':<15}{'Estimated weight memory':>25}")
print("-" * 40)

for name, parameters_billion, bytes_per_parameter in models:
    memory_gb = parameters_billion * bytes_per_parameter
    print(f"{name:<15}{memory_gb:>20.2f} GB")

print("\nNote: These are approximate weight-only estimates.")
print("Actual memory usage will be higher due to inference overhead.")

Model            Estimated weight memory
----------------------------------------
4B FP16                        8.00 GB
4B INT4                        2.00 GB
7B INT4                        3.50 GB

Note: These are approximate weight-only estimates.
Actual memory usage will be higher due to inference overhead.


### Activity 4 - Answers to the Questions

**1. Your laptop has 8 GB RAM — 4B FP16, 4B INT4, or 7B INT4? Why?**

I would choose **4B INT4** as the more conservative starting point. Its estimated model-weight memory is approximately 2 GB, compared with 8 GB for 4B FP16 and 3.5 GB for 7B INT4. However, even 4B INT4 may be tight on an 8 GB laptop because the operating system and inference runtime also need memory. A smaller quantized model may be more practical.

**2. What else besides model weights uses memory?**

Additional memory is needed for the operating system, inference framework, activations, KV cache, tokenization, temporary computation buffers, and other running applications. The KV cache can grow as the context and generated sequence become longer.

**3. Would quantization always be the safer choice? Why or why not?**

No. Quantization can substantially reduce memory requirements and may improve inference speed, but it can also reduce model accuracy or output quality. Support for a particular quantization format depends on the model and inference software. The best choice balances memory availability, performance, compatibility, and output quality.

### Activity 4 - Conclusion

The local computer has 15.68 GB of total RAM, with 1.91 GB available at the time of inspection. PyTorch did not detect a CUDA-compatible GPU. The memory estimates show that model size and numerical precision significantly affect the resources required for local inference.

Quantization can reduce weight memory, but additional runtime memory must also be considered. A model's practical suitability depends on both total hardware capacity and the memory available when the model is executed.

## Activity 5 - Run an LLM Locally Using Ollama

### Objective
To install Ollama, run a language model locally, and understand its internet requirements, inference location, data privacy, and memory limitations.

### Explanation
Ollama is a tool for running supported language models on a local computer. After a model has been downloaded, inference can generally run locally without an internet connection. The model size and quantization affect the memory required to run it.

In [10]:
# Step 5.2: Check Ollama installation

import subprocess

try:
    result = subprocess.run(
        ["ollama", "--version"],
        capture_output=True,
        text=True,
        timeout=15
    )

    if result.returncode == 0:
        print("Ollama is installed.")
        print(result.stdout.strip() or result.stderr.strip())
    else:
        print("Ollama command returned an error:")
        print(result.stderr.strip())

except FileNotFoundError:
    print("Ollama is not installed or is not available in PATH.")

except subprocess.TimeoutExpired:
    print("The Ollama version check timed out.")

Ollama is installed.
ollama version is 0.40.7


In [11]:
# Step 5.3: List models already available locally

try:
    result = subprocess.run(
        ["ollama", "list"],
        capture_output=True,
        text=True,
        timeout=20
    )

    print(result.stdout if result.stdout else result.stderr)

except FileNotFoundError:
    print("Ollama is not available. Complete Step 5.2 first.")

except subprocess.TimeoutExpired:
    print("The Ollama command timed out.")

NAME            ID              SIZE      MODIFIED       
qwen2.5:0.5b    a8b0c5157701    397 MB    48 minutes ago    



In [12]:
# Step 5.6: Recheck available RAM before local inference

import psutil

memory = psutil.virtual_memory()

print("Total RAM:", round(memory.total / (1024 ** 3), 2), "GB")
print("Available RAM:", round(memory.available / (1024 ** 3), 2), "GB")
print("RAM usage:", memory.percent, "%")

Total RAM: 15.68 GB
Available RAM: 1.04 GB
RAM usage: 93.4 %


### Step 5.13 - Local LLM Execution Result

**Tool:** Ollama  
**Model:** `qwen2.5:0.5b`  
**Environment:** Windows 11 laptop

**Procedure:**

1. Installed Ollama on Windows.
2. Downloaded the `qwen2.5:0.5b` model.
3. Started the model using `ollama run qwen2.5:0.5b`.
4. Entered a prompt asking the model to explain artificial intelligence in two sentences.
5. Observed the generated response.

**Observation:** The model was executed through Ollama on the local computer. The initial download required an internet connection. Once downloaded, the model can generally perform inference without internet access, provided the local Ollama service and model files are available.

**Result:** Ollama was installed and used to run a small language model locally.

### Activity 5 - Answers to the Questions

**1. Does the model need internet after downloading?**

No. Once the model has been downloaded, Ollama can generally run it without an internet connection. Internet access is needed for downloading models and for online features or services that the user chooses to use.

**2. Where is inference actually happening?**

Inference happens on the local computer through the Ollama runtime. The downloaded model processes the prompt using the computer's available CPU and compatible hardware acceleration, if supported.

**3. Is any data leaving your machine?**

For a normal local Ollama inference request, the prompt is processed locally rather than sent to a hosted model provider. However, other applications, online integrations, or tools can transmit data if configured to do so. Therefore, local inference reduces data exposure but does not guarantee that all computer activity remains offline.

**4. What happens if the model is too large for your memory?**

The model may fail to load, become very slow, use substantial swap memory, or cause other applications to become unresponsive. Reducing the model size, choosing a quantized model, closing unnecessary applications, or using a computer with more memory can help.

### Conclusion

Ollama was installed on Windows, and a small language model was downloaded for local inference. This activity demonstrated how a language model can run on a personal computer without sending every prompt to a hosted inference provider.

## Activity 6 - Use an LLM Through an API / Inference Provider

### Objective
To call a language model through a hosted inference API and understand how requests are processed, who provides the computing resources, and how API-based inference compares with local execution.

### Explanation
In hosted inference, a program sends a request containing a prompt to a provider's server. The provider runs the model on its infrastructure and returns the generated response. Access may require authentication, and usage limits or charges depend on the provider and service plan.

In [13]:
# Step 6.2: Install the Hugging Face API dependencies

%pip install -q huggingface_hub python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:

# Step 6.3: Load the Groq API key from .env

import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
load_dotenv(env_path)

groq_api_key = os.getenv("GROQ_API_KEY")

if groq_api_key:
    print("Groq API key loaded successfully.")
else:
    print("Token not found. Check the location and contents of your .env file.")


Groq API key loaded successfully.


In [15]:
# Step 6.4: Generate text using the Groq API

from huggingface_hub import InferenceClient

client = InferenceClient(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

try:
    result = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": "Explain artificial intelligence in two simple sentences."
            }
        ],
        max_tokens=150
    )

    print("Provider: Groq")
    print("Model: openai/gpt-oss-20b")
    print("Generated response:")
    print(result.choices[0].message.content)

except Exception as error:
    print("Error type:", type(error).__name__)
    print("Error details:", str(error))

Provider: Groq
Model: openai/gpt-oss-20b
Generated response:
Artificial intelligence is the ability of computers or machines to perform tasks that normally require human intelligence, like learning, reasoning, or recognizing patterns. It works by processing huge amounts of data and adjusting its behavior based on what it learns, so it can make decisions, solve problems, or understand language.


## Activity 6 - Answers to the Questions

**1. What happens between the program and the model?**

The Python program sends the prompt to Groq's hosted API over the internet. Groq processes the request using its inference infrastructure and returns the generated text to the program.

**2. Who pays for the computation?**

The API provider operates the infrastructure. Usage is subject to the provider's pricing, rate limits, and any free-tier allowance available to the account. The API request does not require the model weights to be downloaded to my computer.

**3. What are the advantages and disadvantages compared with local inference?**

- **Advantages:** No need to load a large model locally, less local RAM and GPU usage, and convenient access to hosted models.
- **Disadvantages:** Requires an internet connection, may have usage limits or costs, and sends prompts to an external service, so privacy policies must be considered.

### Observation

I successfully called the Groq API from Python using the `openai/gpt-oss-20b` model. The API returned generated text in the response object, demonstrating remote model inference.


---

## Activity 7  Compare Local vs Colab vs API

**Objective:** Compare three approaches to running large language models: local inference on a laptop, cloud inference using Google Colab GPU, and hosted inference through an API.

**Task:** Compare the three approaches based on hardware requirements, model size, cost, privacy, internet connectivity, setup difficulty, and scalability.


In [16]:

# Activity 7: Compare Local vs Colab vs API
# Based on the experiments performed in Activities 3-6

print("Activity 7: Experimental Comparison")
print("=" * 105)

headers = [
    "Criteria",
    "Local Laptop",
    "Google Colab",
    "Groq API"
]

rows = [
    [
        "Model tested",
        "Qwen2.5:0.5b (Ollama)",
        "distilgpt2",
        "openai/gpt-oss-20b"
    ],
    [
        "Execution location",
        "My Windows laptop",
        "Google cloud",
        "Groq cloud"
    ],
    [
        "Hardware",
        "15.68 GB RAM; no CUDA GPU",
        "NVIDIA Tesla T4",
        "Provider-managed hardware"
    ],
    [
        "GPU memory",
        "No CUDA GPU detected",
        "14.56 GB reported",
        "Not measured locally"
    ],
    [
        "Generation test",
        "Local response tested",
        "20, 50, 100 new tokens",
        "API response received"
    ],
    [
        "Generation time",
        "Not recorded",
        "0.158, 0.395, 0.804 sec",
        "Not recorded"
    ],
    [
        "Peak GPU memory",
        "No CUDA GPU detected",
        "324.52, 325.57, 327.33 MB",
        "Not measured locally"
    ],
    [
        "Internet",
        "Not needed after download",
        "Required",
        "Required"
    ],
    [
        "Cost",
        "Hardware and electricity",
        "Free-tier availability varies",
        "Account limits and pricing apply"
    ],
    [
        "Privacy",
        "Local prompts stay local",
        "Processing occurs in cloud",
        "Prompt sent to Groq"
    ],
    [
        "Scalability",
        "Limited by laptop resources",
        "Limited by session resources",
        "Depends on provider limits"
    ]
]

print(f"{headers[0]:<22} {headers[1]:<28} {headers[2]:<34} {headers[3]}")
print("-" * 125)

for row in rows:
    print(
        f"{row[0]:<22} {row[1]:<28} "
        f"{row[2]:<34} {row[3]}"
    )


Activity 7: Experimental Comparison
Criteria               Local Laptop                 Google Colab                       Groq API
-----------------------------------------------------------------------------------------------------------------------------
Model tested           Qwen2.5:0.5b (Ollama)        distilgpt2                         openai/gpt-oss-20b
Execution location     My Windows laptop            Google cloud                       Groq cloud
Hardware               15.68 GB RAM; no CUDA GPU    NVIDIA Tesla T4                    Provider-managed hardware
GPU memory             No CUDA GPU detected         14.56 GB reported                  Not measured locally
Generation test        Local response tested        20, 50, 100 new tokens             API response received
Generation time        Not recorded                 0.158, 0.395, 0.804 sec            Not recorded
Peak GPU memory        No CUDA GPU detected         324.52, 325.57, 327.33 MB          Not measured locally


### Activity 7 - Answers to the Questions

**1. If you had an 8 GB RAM laptop with no GPU, which approach would you pick?**

I would choose Google Colab when a suitable GPU runtime is available, because the computation runs on cloud hardware rather than relying on my laptop's limited RAM. I would choose an API such as Groq when I need a convenient way to generate responses without managing a model runtime. A small quantized model is another option for local experimentation, although performance may be slow.

**2. Which approach is most private? Which is most scalable?**

- **Most private:** Local inference, provided the application and model run locally and do not send data to external services.
- **Most scalable:** A hosted API can scale more easily than a personal laptop, subject to provider capacity, rate limits, and service terms.

**3. When would you choose local over cloud, or the reverse?**

- **Choose local inference** when working with sensitive information, needing offline operation after downloading the model, or experimenting without sending prompts to a third party.
- **Choose cloud GPU inference** when you need GPU resources for experiments but do not have a suitable local GPU.
- **Choose API inference** when you want a simple integration, access to provider-hosted models, or an application that may serve multiple users.

### Conclusion

Local inference offers greater control over data and offline use, cloud GPUs provide computing resources for experimentation, and hosted APIs simplify access to remote models. The best choice depends on hardware, privacy, cost, connectivity, and scalability requirements.


---

## Activity 8 - Temperature Experiment

**Objective:** Observe how temperature affects the diversity and predictability of text generated by a language model.

**Task:** Generate text using the same prompt at temperatures `0.1`, `0.7`, and `1.0`. Run each setting three times and compare the results.

**Prompt:** `A useful programming language for beginners is`


In [17]:

# Activity 8: Load a text-generation model

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_name = "distilbert/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = 0 if torch.cuda.is_available() else -1

local_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("Model loaded successfully:", model_name)
print("Execution device:", "GPU" if device == 0 else "CPU")


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 389.29it/s]

Model loaded successfully: distilbert/distilgpt2
Execution device: CPU


In [18]:

# Activity 8: Temperature Experiment

prompt = "A useful programming language for beginners is"
temperatures = [0.1, 0.7, 1.0]

print("Prompt:", prompt)
print("=" * 70)

for temp in temperatures:
    print(f"\n--- Temperature: {temp} ---")

    for run in range(3):
        result = local_generator(
            prompt,
            max_new_tokens=40,
            do_sample=True,
            temperature=temp,
            top_p=0.95,
            num_return_sequences=1,
            pad_token_id=local_generator.tokenizer.eos_token_id
        )

        generated_text = result[0]["generated_text"]
        generated_part = generated_text[len(prompt):].strip()

        print(f"Run {run + 1}: {generated_part}")

print("\nExperiment completed.")


Prompt: A useful programming language for beginners is

--- Temperature: 0.1 ---
Run 1: the C++ language. It is a great tool for beginners to learn C++ and C++. It is also a great tool for beginners to learn C++ and C++. It is also a
Run 2: the C++ language. It is a language that is used to build and run programs in C++. It is also a language that is used to build and run programs in C++. It is
Run 3: the C++ language. It is a great tool for beginners to learn C++.

--- Temperature: 0.7 ---
Run 1: Java. It can be found at http://jquery.com/Jquery.
Run 2: the Python programming language for Python and has a few interesting features.
Run 3: JavaScript. This article is aimed at beginner programmers who are familiar with JavaScript and want to learn more.




The most common language used for a beginner is JavaScript. A beginner language such

--- Temperature: 1.0 ---
Run 1: the PHP compiler. You can use a script to create code using PHP.php:
Run 2: CodeServe. It provides the core language

### Activity 8 - Answers to the Questions

**1. Which output felt most predictable? Most creative?**

- **Most predictable:** Temperature 0.1. The three outputs repeatedly mentioned C++ and learning programming, with similar wording.
- **Most creative or varied:** Temperature 1.0. The outputs mentioned different programming languages and concepts, including Haskell, Ruby, and Python. However, some responses were less coherent.
- **Temperature 0.7:** The outputs varied considerably, ranging from very short responses such as “this:” to a reference to Windows or OS X.

**2. Does temperature change what the model knows?**

No. Temperature does not change the model's learned knowledge or its parameters. It changes how the model selects the next token from its predicted probability distribution when sampling is enabled. Therefore, the generated response may differ even though the underlying model remains the same.

**3. When would you want low temperature? High temperature?**

- **Low temperature (e.g., 0.1):** I would use it for predictable, consistent responses, such as answering straightforward questions or generating structured text.
- **High temperature (e.g., 1.0):** I would use it for brainstorming, creative writing, or exploring different ideas, while checking the output for relevance and correctness.

### Conclusion

The experiment showed that temperature affects the predictability and variety of generated text. Lower temperature produced more consistent outputs in this experiment, while higher temperature produced more varied responses. Temperature changes the generation process, not the model's learned knowledge.


---

## Activity 9 - Build a Mini Chatbot with Conversation History

**Objective:** Build a simple chatbot that maintains conversation history and uses previous messages as context for later responses.

**Task:** Create a chatbot using Ollama or an API. Ask at least two related questions and observe whether the chatbot can use information from the earlier messages.


In [ ]:

# Activity 9: Mini Chatbot with Conversation History

import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "qwen2.5:0.5b"

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Answer clearly and concisely."
    }
]

print("Mini Chatbot started. Type 'exit' to stop.")
print("-" * 55)

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Chatbot: Goodbye!")
        break

    if not user_input:
        continue

    messages.append({
        "role": "user",
        "content": user_input
    })

    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL_NAME,
                "messages": messages,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()

        reply = response.json()["message"]["content"]
        print("Chatbot:", reply)

        messages.append({
            "role": "assistant",
            "content": reply
        })

    except requests.RequestException as error:
        print("Request failed:", error)
        messages.pop()
        break

print("\nTotal conversation messages retained:", len(messages))


Mini Chatbot started. Type 'exit' to stop.
-------------------------------------------------------


### Activity 9 - Answers to the Questions

**1. Why does the chatbot need to remember previous messages?**

The chatbot needs conversation history to understand the context of the current question. In my experiment, I first provided my name and course, and the chatbot later answered questions about them using the retained conversation history.

**2. What happens if you drop the conversation history?**

If the conversation history is removed, the chatbot receives only the latest message and may lose access to information provided in earlier turns. For example, without the initial message containing my name, the chatbot may not know the answer to “What is my name?”

**3. What's the difference between this and a single one-off prompt?**

A single one-off prompt is processed independently and does not automatically include earlier messages. A chatbot loop maintains conversation history and sends relevant previous messages with each new request, allowing the model to respond with greater continuity and context.

### Observation

The Ollama chatbot successfully responded to follow-up questions about my name and course. The Python program retained seven messages: one system message, three user messages, and three assistant responses.

### Conclusion

Conversation history enables a chatbot to maintain context across multiple interactions. Without it, each request may lack important information from previous turns.


---

## Activity 10 - Customize the Chatbot's Role

**Objective:** Customize a chatbot's behavior by changing its system instructions and observing how the instructions affect its responses.

**Task:** Modify the chatbot from Activity 9 to assign it a specific role. Test the customized chatbot with a question and compare its responses with the original chatbot.



In [ ]:

# Activity 10: Customize the Chatbot's Role

import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "qwen2.5:0.5b"

messages = [
    {
        "role": "system",
        "content": (
            "You are a friendly Java programming tutor for MCA students. "
            "Explain Java concepts in simple language, use short examples "
            "when helpful, and guide learners step by step."
        )
    }
]

print("Java Tutor Chatbot started.")
print("Type 'exit' to stop.")
print("-" * 55)

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Chatbot: Goodbye! Keep practicing Java.")
        break

    if not user_input:
        continue

    messages.append({
        "role": "user",
        "content": user_input
    })

    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL_NAME,
                "messages": messages,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()

        reply = response.json()["message"]["content"]
        print("Java Tutor:", reply)

        messages.append({
            "role": "assistant",
            "content": reply
        })

    except requests.RequestException as error:
        print("Request failed:", error)
        messages.pop()
        break

print("\nTotal conversation messages retained:", len(messages))


Java Tutor Chatbot started.
Type 'exit' to stop.
-------------------------------------------------------
Java Tutor: A Java class is a blueprint for creating objects, which are instances of that class. It serves as a template for how objects should be constructed and what operations should be performed on them. Here's a simple example to illustrate what a Java class is:

```java
public class Student {
    private String name;
    private int age;

    // Constructor
    public Student(String name, int age) {
        this.name = name;
        this.age = age;
    }

    // Methods
    public void displayInfo() {
        System.out.println("Name: " + name);
        System.out.println("Age: " + age);
    }

    // Main method
    public static void main(String[] args) {
        Student student = new Student("John Doe", 20);
        student.displayInfo();
    }
}
```

In this example, the `Student` class has two fields: `name` and `age`. The constructor is defined where it initializes these

### Activity 10 - Answers to the Questions

**1. How did the responses change compared to Activity 9?**

In Activity 9, the chatbot acted as a general-purpose assistant and answered questions about my name and course using conversation history. In Activity 10, it acted as a Java programming tutor, explaining Java classes with a `Student` class example and discussing programming concepts such as inheritance. Its responses became more focused on programming education.

**2. Could the same model behave like a completely different assistant just from the instructions you gave it?**

Yes. The same `qwen2.5:0.5b` model behaved differently when I changed its system instructions. In Activity 9, it followed general assistant instructions, while in Activity 10, it followed instructions to act as a friendly Java tutor. The underlying model remained the same, but the system message guided its role, response style, and focus.

**3. What role would you build next, and why?**

I would build a Java interview preparation coach. It could ask technical interview questions, explain answers in simple language, identify gaps in my understanding, and provide follow-up questions. This role would help students strengthen their Java knowledge and prepare for software developer interviews.

### Conclusion

This activity demonstrated that system instructions can guide the same language model to perform different conversational roles. A carefully designed role can make responses more relevant to a particular task or user need.
